# RiskyPy runs on Google Colab

This notebook runs the code generation experiment of the RiskyPy repository on a Colab GPU runtime (A100 80 GB) and saves the repository, with the generated runs, both in Google Drive and on the local machine.

Steps:

1. Upload the zip of the repository.
2. Connect Google Drive.
3. Install the dependencies of the experiment.
4. Run `experiment/run_experiment.py`, which writes the run files in `experiment/runs`.
5. Save the repository with its runs as a zip, in Google Drive and on the local machine.

## 1. Upload the repository

Select the zip of the repository. It is extracted in `/content/repository`, and the repository folder is the one that contains `experiment/run_experiment.py`.

In [ ]:
import shutil
import zipfile
from pathlib import Path
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one zip file.")
zip_name = next(iter(uploaded))

WORK_DIR = Path("/content/repository")
shutil.rmtree(WORK_DIR, ignore_errors=True)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(WORK_DIR)
script = next(WORK_DIR.rglob("experiment/run_experiment.py"), None)
if script is None:
    raise FileNotFoundError("experiment/run_experiment.py not found in the uploaded zip.")
REPO_DIR = script.parent.parent
print(f"Repository: {REPO_DIR}")

## 2. Connect Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Install the dependencies

In [ ]:
%pip install -q -r "{REPO_DIR}/experiment/requirements.txt"

## 4. Run the experiment

Generates the six runs (two models, three approaches) in `experiment/runs` of the repository. A run whose file is already in the uploaded repository is skipped. If the script stops with an error, the runs completed so far remain in `experiment/runs` and can still be saved with the last cell.

In [ ]:
import runpy

# The returned namespace of the script is assigned, so that the notebook does not try to display it.
_ = runpy.run_path(str(REPO_DIR / "experiment" / "run_experiment.py"), run_name="__main__")

## 5. Save the repository with the runs

The repository, with its `experiment/runs` folder, is zipped (without Python cache folders) as `<repository>_with_runs.zip`. The zip is copied to the root of "My Drive" (replacing a file of the same name) and downloaded to the local machine.

In [ ]:
OUTPUT_ZIP = Path("/content") / f"{REPO_DIR.name}_with_runs.zip"
with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(REPO_DIR.rglob("*")):
        if path.is_file() and "__pycache__" not in path.parts:
            archive.write(path, path.relative_to(REPO_DIR.parent))
print("Runs:", sorted(p.name for p in (REPO_DIR / "experiment" / "runs").glob("*.json")))

shutil.copy(OUTPUT_ZIP, "/content/drive/MyDrive/")
drive.flush_and_unmount()
print(f"Saved in Google Drive: MyDrive/{OUTPUT_ZIP.name}")

files.download(str(OUTPUT_ZIP))